# Reels Stüdyo — Ücretsiz Colab Video Motoru\nÇalışma zamanı > Çalışma zamanı türünü değiştir > **T4 GPU** seçin. Sonra aşağıdaki hücreleri sırayla çalıştırın. Son hücrede çıkan `https://...gradio.live` adresini Reels Stüdyo'ya yapıştırın. Ücretsiz Colab GPU kotası ve erişimi garanti değildir. Modelin ilk indirilmesi zaman alabilir. Çıktı düşük çözünürlüklü, kısa bir yapay zekâ videosudur; yüksek gerçekçilik garanti edilmez.\n

In [ ]:
!pip -q install 'diffusers==0.31.0' 'transformers<4.46' 'accelerate>=0.33' 'gradio>=5,<6' 'imageio[ffmpeg]' 'huggingface_hub<1.0'\n

In [ ]:
import os, gc, torch, imageio, tempfile
from diffusers import DiffusionPipeline, DPMSolverMultistepScheduler
import gradio as gr
from fastapi.middleware.cors import CORSMiddleware
MODEL = "damo-vilab/text-to-video-ms-1.7b"
print("Model indiriliyor. İlk çalıştırma uzun sürebilir.")
pipe = DiffusionPipeline.from_pretrained(MODEL, torch_dtype=torch.float16, variant="fp16")
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
pipe.enable_model_cpu_offload()
pipe.enable_vae_slicing()
def generate(prompt, seconds=2):
    if not prompt or not prompt.strip(): raise gr.Error("Sahne açıklaması yaz.")
    seconds = max(1, min(3, int(seconds)))
    frames = pipe(prompt.strip(), num_inference_steps=20, num_frames=seconds*8, height=256, width=256).frames[0]
    path = tempfile.NamedTemporaryFile(suffix=".mp4", delete=False).name
    imageio.mimsave(path, frames, fps=8, macro_block_size=1)
    gc.collect()
    torch.cuda.empty_cache()
    return path
demo=gr.Interface(fn=generate, inputs=[gr.Textbox(label="Sahne",lines=3),gr.Slider(1,3,value=2,step=1,label="Saniye")],outputs=gr.Video(label="Üretilen video"),title="Reels Stüdyo — Colab Video Motoru")
demo.app.add_middleware(CORSMiddleware,allow_origins=["https://filorensuarez.github.io"],allow_methods=["*"],allow_headers=["*"])
demo.launch(share=True,debug=True)
